# Agentic RAG（讓 Agent 主動規劃檢索）

## 模組脈絡：從固定檢索流程，升級成可判斷、可重試的知識流程

本筆記隸屬 **04-知識收斂**。前面的 RAG 多半是固定管線：使用者問題 → embedding 檢索 → 塞 context → 回答。Agentic RAG 則把檢索變成 agent 可控制的行動：先判斷是否需要檢索、如何改寫查詢、是否要二次檢索、最後再檢查答案是否有根據。

> 這章補上「RAG + Agent」的交界：仍以知識收斂為主，但已經開始接近 05 的 agent harness。

## 1. Vanilla RAG vs Agentic RAG

| 流程 | Vanilla RAG | Agentic RAG |
|------|-------------|-------------|
| 檢索時機 | 每次都檢索 | 先判斷是否需要檢索 |
| query | 直接用原問題 | 可改寫、拆解、產生多個 query |
| 失敗處理 | 通常直接回答 | 可重試、換 query、要求更多資料 |
| 回答檢查 | 靠 prompt 約束 | 額外做 groundedness / sufficiency check |
| 代價 | 便宜、穩定 | 較貴、較慢、但複雜問題更準 |

Agentic RAG 不一定比較好。若問題很單純、文件很乾淨，vanilla RAG 通常更穩。當問題需要拆解、多跳推理、跨文件比較、或第一次檢索可能失敗時，才值得使用 agentic 流程。

## 2. Agentic RAG 的基本控制迴圈

一個實務上常見的 agentic RAG pipeline：

1. **Route**：判斷問題是否需要外部知識。
2. **Plan / Rewrite**：把問題改寫成適合檢索的 query，必要時拆成多個子問題。
3. **Retrieve**：執行檢索。
4. **Evaluate Context**：判斷 context 是否足夠回答。
5. **Retry or Answer**：不足就換 query 重試；足夠就根據 context 回答。
6. **Grounding Check**：檢查答案是否每個關鍵主張都有來源。

關鍵控制點：一定要有 `max_steps`，否則 agent 可能一直改 query、一直檢索。

In [ ]:
# 本章用一個小型 in-memory corpus 示範 agentic RAG 控制流程。
# 真實專案可把 retrieve() 換成 Chroma、Elasticsearch、OpenAI file_search、或企業搜尋 API。

from dataclasses import dataclass
from typing import Literal
import math
import re


@dataclass
class Document:
    doc_id: str
    title: str
    text: str


docs = [
    Document(
        "rag-001",
        "RAG 基礎",
        "RAG 先檢索外部文件，再把相關 context 放進 prompt，降低模型憑空回答的機率。",
    ),
    Document(
        "rag-002",
        "Advanced RAG",
        "進階 RAG 常見技巧包含 query expansion、HyDE、reranking、small-to-big retrieval。",
    ),
    Document(
        "rag-003",
        "Agentic RAG",
        "Agentic RAG 讓 agent 判斷是否檢索、改寫查詢、重試檢索，並檢查答案是否被 context 支撐。",
    ),
    Document(
        "agent-001",
        "Agent Harness",
        "Agent harness 需要工具邊界、迴圈上限、權限控管與稽核紀錄，避免工具濫用與無限迴圈。",
    ),
]

In [ ]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z0-9]+|[\u4e00-\u9fff]", text.lower())


def score(query: str, doc: Document) -> float:
    q_terms = tokenize(query)
    d_terms = tokenize(doc.title + " " + doc.text)
    if not q_terms or not d_terms:
        return 0.0
    overlap = sum(1 for term in q_terms if term in d_terms)
    return overlap / math.sqrt(len(q_terms) * len(d_terms))


def retrieve(query: str, k: int = 2) -> list[tuple[Document, float]]:
    ranked = sorted(((doc, score(query, doc)) for doc in docs), key=lambda x: x[1], reverse=True)
    return [(doc, s) for doc, s in ranked[:k] if s > 0]


retrieve("agentic rag 如何避免檢索失敗")

## 3. Route：先判斷需不需要檢索

不是每個問題都需要 RAG。像「幫我改寫這句話」通常不需要外部知識；像「根據公司手冊回答請假規則」就需要。這個 route 可以用規則、分類模型、或 LLM function calling。

In [ ]:
RouteDecision = Literal["answer_directly", "retrieve"]


def route_question(question: str) -> RouteDecision:
    retrieval_keywords = ["根據", "文件", "資料", "課程", "手冊", "規定", "rag", "agent"]
    if any(keyword.lower() in question.lower() for keyword in retrieval_keywords):
        return "retrieve"
    return "answer_directly"


for q in ["請把這句話改得正式一點", "Agentic RAG 和 advanced RAG 差在哪？"]:
    print(q, "=>", route_question(q))

## 4. Rewrite：把使用者問題改成檢索 query

使用者問題常包含口語、省略、代名詞或多個意圖。Agentic RAG 會先改寫成更適合搜尋的 query。正式系統可以用 LLM 產生 2-3 個 query，再合併檢索結果。

In [ ]:
def rewrite_queries(question: str) -> list[str]:
    q = question.strip()
    queries = [q]
    if "agentic" in q.lower() and "rag" in q.lower():
        queries.append("Agentic RAG route rewrite retrieve evaluate retry groundedness")
        queries.append("RAG agent query rewrite context sufficiency check")
    return queries[:3]


rewrite_queries("Agentic RAG 和 advanced RAG 差在哪？")

## 5. Context Sufficiency：判斷資料夠不夠

Agentic RAG 的重點不是「多檢索幾次」，而是每次檢索後做決策：目前 context 是否足夠？若不足，是換 query、找其他工具，還是直接承認資料不足？

In [ ]:
def is_context_sufficient(question: str, retrieved: list[tuple[Document, float]]) -> bool:
    if not retrieved:
        return False
    best_score = retrieved[0][1]
    joined = " ".join(doc.text for doc, _ in retrieved).lower()
    important_terms = [term for term in tokenize(question) if len(term) > 1]
    covered_terms = sum(1 for term in important_terms if term in joined)
    return best_score >= 0.08 and covered_terms >= max(1, len(important_terms) // 4)


question = "Agentic RAG 和 advanced RAG 差在哪？"
hits = retrieve(question)
print(hits)
print("sufficient:", is_context_sufficient(question, hits))

## 6. 組合成一個 Agentic RAG Loop

下面的 `agentic_rag()` 仍是簡化版，但保留真實系統最重要的控制點：route、rewrite、多次 retrieve、sufficiency check、max_steps。

In [ ]:
def synthesize_answer(question: str, retrieved: list[tuple[Document, float]]) -> str:
    if not retrieved:
        return "目前資料不足，無法根據文件回答。"

    bullets = []
    for doc, _ in retrieved:
        bullets.append(f"- [{doc.doc_id}] {doc.text}")
    context = "\n".join(bullets)
    return (
        f"根據檢索內容，回答問題：{question}\n\n"
        f"可用依據：\n{context}\n\n"
        "教學重點：正式系統應把這段 context 交給 LLM 產生自然語言答案，並要求引用 doc_id。"
    )


def agentic_rag(question: str, max_steps: int = 3) -> dict:
    if route_question(question) == "answer_directly":
        return {"mode": "direct", "answer": "這題不需要檢索，可直接由模型處理。"}

    trace = []
    all_hits: list[tuple[Document, float]] = []
    for step, query in enumerate(rewrite_queries(question), start=1):
        if step > max_steps:
            break
        hits = retrieve(query, k=2)
        trace.append({"step": step, "query": query, "hits": [(doc.doc_id, round(s, 3)) for doc, s in hits]})
        all_hits.extend(hits)
        deduped = {doc.doc_id: (doc, s) for doc, s in all_hits}
        current_hits = sorted(deduped.values(), key=lambda x: x[1], reverse=True)[:3]
        if is_context_sufficient(question, current_hits):
            return {"mode": "rag", "trace": trace, "answer": synthesize_answer(question, current_hits)}

    deduped = {doc.doc_id: (doc, s) for doc, s in all_hits}
    current_hits = sorted(deduped.values(), key=lambda x: x[1], reverse=True)[:3]
    return {"mode": "rag_insufficient", "trace": trace, "answer": synthesize_answer(question, current_hits)}


result = agentic_rag("Agentic RAG 和 advanced RAG 差在哪？")
result

## 7. 若接上 LLM，Prompt 應該怎麼寫？

正式系統常把上面的 route / rewrite / sufficiency check 改成結構化輸出。重點是讓模型輸出可被程式控制的 JSON，而不是自由文字。

In [ ]:
route_prompt = """
你是 RAG router。請判斷使用者問題是否需要外部知識。

只輸出 JSON：
{
  "decision": "answer_directly" | "retrieve",
  "reason": "一句話理由"
}

使用者問題：{question}
"""

rewrite_prompt = """
你是 query rewrite agent。請把問題改寫成 1-3 個適合檢索的 query。

只輸出 JSON：
{
  "queries": ["query 1", "query 2"]
}

使用者問題：{question}
"""

check_prompt = """
你是 RAG groundedness checker。請判斷 context 是否足夠回答問題。

只輸出 JSON：
{
  "sufficient": true | false,
  "missing": ["缺少的資訊"]
}

問題：{question}
Context：{context}
"""

print(route_prompt[:160])

## 8. 設計檢查表

做 Agentic RAG 時，先檢查這些點：

- 是否真的需要 agentic？能用 vanilla RAG 解決就不要增加複雜度。
- 是否有 `max_steps` / `max_tool_calls`？
- 是否記錄每次 query、檢索結果、答案引用？
- context 不足時，是重試、換工具，還是明確拒答？
- 是否防範間接 prompt injection？文件內容不能命令 agent 改系統規則。
- 評估時是否分開看 retrieval quality、answer groundedness、end-to-end answer quality？

---

## 本章小結

1. Agentic RAG 是把 RAG 的固定管線，改成 agent 可判斷與可重試的控制流程。
2. 核心步驟是 route、rewrite、retrieve、sufficiency check、answer、grounding check。
3. 它適合多跳、跨文件、檢索容易失敗的問題；簡單 FAQ 不一定需要。
4. 工程上務必加 `max_steps`、稽核 trace、引用來源與不足拒答策略。